In [ ]:
%matplotlib inline
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/data1")
print(DATA_DIR.exists(), DATA_DIR)

In [ ]:
CLASSES = ["idle", "walking", "running", "stairs"]

In [ ]:
def read_acc_csv(fp: Path) -> pd.DataFrame:
    """
    Reads a single accelerometer CSV.
    Tries to infer 3 accelerometer columns (ax, ay, az).
    Works with/without header; drops non-numeric columns.
    """

    df = pd.read_csv(fp)

    # ????????? ???? ??????? ???????
    num = df.select_dtypes(include=[np.number]).copy()


    if num.shape[1] < 3:
        df2 = pd.read_csv(fp, header=None)
        num2 = df2.select_dtypes(include=[np.number]).copy()
        num = num2

    # ?????? ????? 3 ??????? ??????? ?? ????????????
    if num.shape[1] < 3:
        raise ValueError(f"File {fp} has <3 numeric columns. Can't infer ax,ay,az.")

    num = num.iloc[:, :3].copy()
    num.columns = ["ax", "ay", "az"]

    # ????????? NaN/inf
    num = num.replace([np.inf, -np.inf], np.nan).dropna()

    return num

In [ ]:
def load_dataset(data_dir: Path, classes: list[str]) -> pd.DataFrame:
    """
    Loads all CSV files from class folders into a long dataframe:
    columns: [ax, ay, az, label, file_id]
    """
    rows = []
    for label in classes:
        folder = data_dir / label
        if not folder.exists():
            raise FileNotFoundError(f"Folder not found: {folder}")

        files = sorted(folder.glob("*.csv"))
        if len(files) == 0:
            raise FileNotFoundError(f"No CSV files found in {folder}")

        for fp in files:
            sig = read_acc_csv(fp)
            sig["label"] = label
            sig["file_id"] = fp.name
            rows.append(sig)

    return pd.concat(rows, ignore_index=True)

In [ ]:
def windowize(df_long: pd.DataFrame, window_size: int = 64, step: int = 32, pad_short: bool = True):
    """
    Splits signals into windows per file.
    If a file is shorter than window_size and pad_short=True, it will be padded to create 1 window.
    Returns:
      X_windows: [n_windows, window_size, 3]
      y_windows: [n_windows]
    """
    X_list, y_list = [], []

    for (label, file_id), part in df_long.groupby(["label", "file_id"]):
        arr = part[["ax", "ay", "az"]].values
        n = arr.shape[0]

        if n < window_size:
            if not pad_short:
                continue
            # pad by repeating last value
            pad_len = window_size - n
            pad_block = np.repeat(arr[-1:, :], repeats=pad_len, axis=0)
            w = np.vstack([arr, pad_block])
            X_list.append(w)
            y_list.append(label)
            continue

        for start in range(0, n - window_size + 1, step):
            w = arr[start:start + window_size]
            X_list.append(w)
            y_list.append(label)

    return np.array(X_list, dtype=float), np.array(y_list)

In [ ]:
def _entropy(x: np.ndarray, bins: int = 16) -> float:
    """Shannon entropy of a 1D signal (hist-based)."""
    hist, _ = np.histogram(x, bins=bins, density=True)
    hist = hist[hist > 0]
    if len(hist) == 0:
        return 0.0
    return float(-np.sum(hist * np.log2(hist)))

def _zero_cross_rate(x: np.ndarray) -> float:
    """Zero-crossing rate."""
    return float(np.mean((x[:-1] * x[1:]) < 0))

def _energy(x: np.ndarray) -> float:
    """Mean energy of a 1D signal."""
    return float(np.mean(x ** 2))

def _sma(ax: np.ndarray, ay: np.ndarray, az: np.ndarray) -> float:
    """Signal magnitude area."""
    return float(np.mean(np.abs(ax) + np.abs(ay) + np.abs(az)))

def extract_features_simple(Xw: np.ndarray) -> pd.DataFrame:
    """
    Simple features: mean/std for each axis + magnitude.
    Xw: [n_windows, window, 3]
    """
    feats = []
    for w in Xw:
        ax, ay, az = w[:, 0], w[:, 1], w[:, 2]
        mag = np.sqrt(ax**2 + ay**2 + az**2)

        f = {
            "ax_mean": np.mean(ax), "ax_std": np.std(ax),
            "ay_mean": np.mean(ay), "ay_std": np.std(ay),
            "az_mean": np.mean(az), "az_std": np.std(az),
            "mag_mean": np.mean(mag), "mag_std": np.std(mag),
        }
        feats.append(f)
    return pd.DataFrame(feats)

def extract_features_tdf(Xw: np.ndarray) -> pd.DataFrame:
    """
    Time-domain features (richer):
    - mean, std, min, max, median, iqr, mad, energy, entropy, zcr
    - magnitude features
    - SMA
    - correlations between axes
    """
    feats = []
    for w in Xw:
        ax, ay, az = w[:, 0], w[:, 1], w[:, 2]
        mag = np.sqrt(ax**2 + ay**2 + az**2)

        def stats(prefix: str, x: np.ndarray) -> dict:
            q75, q25 = np.percentile(x, [75, 25])
            iqr = q75 - q25
            med = np.median(x)
            mad = np.median(np.abs(x - med))
            return {
                f"{prefix}_mean": float(np.mean(x)),
                f"{prefix}_std": float(np.std(x)),
                f"{prefix}_min": float(np.min(x)),
                f"{prefix}_max": float(np.max(x)),
                f"{prefix}_median": float(med),
                f"{prefix}_iqr": float(iqr),
                f"{prefix}_mad": float(mad),
                f"{prefix}_energy": _energy(x),
                f"{prefix}_entropy": _entropy(x),
                f"{prefix}_zcr": _zero_cross_rate(x),
            }

        f = {}
        f.update(stats("ax", ax))
        f.update(stats("ay", ay))
        f.update(stats("az", az))
        f.update(stats("mag", mag))

        # SMA
        f["sma"] = _sma(ax, ay, az)

        # Correlations
        def safe_corr(a, b):
            if np.std(a) < 1e-12 or np.std(b) < 1e-12:
                return 0.0
            return float(np.corrcoef(a, b)[0, 1])

        f["corr_ax_ay"] = safe_corr(ax, ay)
        f["corr_ax_az"] = safe_corr(ax, az)
        f["corr_ay_az"] = safe_corr(ay, az)

        feats.append(f)

    return pd.DataFrame(feats)

In [ ]:
def evaluate_model(name: str, model, X_train, y_train, X_test, y_test, labels_order):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print("=" * 80)
    print(name)
    print("=" * 80)
    print(classification_report(y_test, pred, target_names=labels_order))

    cm = confusion_matrix(y_test, pred, labels=labels_order)
    cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in labels_order],
                         columns=[f"pred_{c}" for c in labels_order])
    display(cm_df)

    acc = accuracy_score(y_test, pred)
    f1_macro = f1_score(y_test, pred, average="macro")
    f1_weighted = f1_score(y_test, pred, average="weighted")

    return {"model": name, "accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted}

In [ ]:
df_long = load_dataset(DATA_DIR, CLASSES)
display(df_long.head())
print("Loaded rows:", len(df_long), "| files:", df_long["file_id"].nunique())

In [ ]:
WINDOW = 64
STEP = 32

Xw, yw = windowize(df_long, window_size=WINDOW, step=STEP, pad_short=True)
print("Windows:", Xw.shape, "Labels:", yw.shape)
print("Label distribution:\n", pd.Series(yw).value_counts())

In [ ]:
X_simple = extract_features_simple(Xw)
X_tdf = extract_features_tdf(Xw)

print("Simple features:", X_simple.shape)
print("TDF features:", X_tdf.shape)

In [ ]:
Xtr_s, Xte_s, ytr, yte = train_test_split(X_simple, yw, test_size=0.2, random_state=42, stratify=yw)
Xtr_t, Xte_t, _, _ = train_test_split(X_tdf,   yw, test_size=0.2, random_state=42, stratify=yw)

labels_order = CLASSES

In [ ]:
svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced", random_state=42))
])

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

In [ ]:
results = []
results.append(evaluate_model("SVM (simple features)", svm, Xtr_s, ytr, Xte_s, yte, labels_order))
results.append(evaluate_model("RandomForest (simple features)", rf, Xtr_s, ytr, Xte_s, yte, labels_order))

In [ ]:
results.append(evaluate_model("SVM (time-domain features)", svm, Xtr_t, ytr, Xte_t, yte, labels_order))
results.append(evaluate_model("RandomForest (time-domain features)", rf, Xtr_t, ytr, Xte_t, yte, labels_order))

In [ ]:
res_df = pd.DataFrame(results).sort_values(by="f1_macro", ascending=False)
display(res_df)

In [ ]:
print("Model ranking by macro F1 (main metric):")
display(res_df)


## ????????

?? ????????? 2 ?????? (SVM ?? RandomForest) ?? 2 ??????? ?????:
1) **Simple features** (mean/std ?? ???? ?? magnitude)
2) **Time-domain features** (????????? ??????????: median, IQR, MAD, energy, entropy, ZCR, SMA, ?????????)

### ?????????? ?? classification_report
- ?? **simple features** ?????? ????? ???? ??? ????? **running** (?????? support ?? ????? ?????????).
- ?? **time-domain features** ?????? ?????? ??????? ???????????; ????????? ????????? ??????? **RandomForest (time-domain features)**.

### ????????
??? ????? ???????? ????????? **time-domain features** ????? ????????? ??????? ?????? (???????? ?? macro F1), ? RandomForest ?? ??? ??????? ?????? ??????????????.
